Troy Healey

In [49]:
# Troy Healey

import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# load datasets
df_movies = pd.read_csv("movies_metadata.csv")
df_reviews = pd.read_csv("synthetic_movie_reviews.csv")
df_users = pd.read_csv("synthetic_users.csv")

# load transformer model
embedder_model = SentenceTransformer("all-MiniLM-L6-v2")
# embed reviews
embeddings = embedder_model.encode(df_reviews["text_review"])
df_reviews["review_embeddings"] = list(embeddings)

movie_embeddings = {}
# group by movie id
groups = df_reviews.groupby("movie_id")
for movie, group in groups:
    # aggregate review embeddings using average
    review_vectors = group["review_embeddings"]
    matrix = np.vstack(review_vectors)
    movie_vector = matrix.mean(axis=0)
    movie_embeddings[movie] = movie_vector


df_reviews["datetime"] = pd.to_datetime(df_reviews["datetime"])
train_rows = []
test_rows = []
excluded_users = []
user_groups = df_reviews.groupby("user_id")
for user, group in user_groups:
    group = group.sort_values("datetime")
    positive_ratings = group[group["rating"] >= 4]
    # exclude user if no positive interaction
    if positive_ratings.empty:
        excluded_users.append(user)
        continue
    # add last positive row to test_rows
    test = positive_ratings.iloc[-1]
    test_rows.append(test)
    # add the rest of the movies to training rows
    training_group = group[group.index != test.name]
    train_rows.append(training_group)

df_train = pd.concat(train_rows).reset_index(drop=True)
df_test = pd.DataFrame(test_rows)
df_test.reset_index(drop=True, inplace=True)

# building user vectors
user_profiles = {}
excluded_profile_users = []

for user, group in df_train.groupby("user_id"):
    positive_train = group[group["rating"] >= 4]
    # exclude user if no positive interaction
    if positive_train.empty:
        excluded_profile_users.append(user)
        continue
    # keep track of movie embedding vectors
    movie_vectors = []
    for movie in positive_train["movie_id"]:
        if movie in movie_embeddings:
            movie_vectors.append(movie_embeddings[movie])
    if len(movie_vectors) == 0:
        excluded_profile_users.append(user)
        continue
    user_profiles[user] = np.vstack(movie_vectors).mean(axis=0)

# create movie matrix
movie_ids = list(movie_embeddings.keys())
movie_matrix = np.vstack([movie_embeddings[movie_id] for movie_id in movie_ids])

# recommend top 5 movies per user
top_k = 5
recommendations = {}

for user, user_profile in user_profiles.items():
    sims = cosine_similarity([user_profile], movie_matrix)[0]
    seen_movies = set(df_train[df_train["user_id"] == user]["movie_id"])

    candidate_scores = []
    for i, movie_id in enumerate(movie_ids):
        # make sure user has not seen movie before
        if movie_id not in seen_movies:
            candidate_scores.append((movie_id, sims[i]))
    candidate_scores.sort(key=lambda x: x[1], reverse=True)
    # recommend top 5 candidate movies
    top_movies = []
    for movie_id, score in candidate_scores[:top_k]:
        top_movies.append(movie_id)
    recommendations[user] = top_movies

# evaluate hit, precision, recall
evaluation_rows = []

for _, row in df_test.iterrows():
    user = row["user_id"]
    true_movie = row["movie_id"]
    if user not in recommendations:
        continue
    # calculate hit, precision, recall
    recs = recommendations[user]
    if true_movie in recs:
        hit = 1
    else:
        hit = 0
    precision = hit / 5
    recall = hit / 1

    evaluation_rows.append({
        "user_id": user,
        "test_movie": true_movie,
        "recommendations": recs,
        "Hit@5": hit,
        "Precision@5": precision,
        "Recall@5": recall
    })

# convert to dataframe
df_eval = pd.DataFrame(evaluation_rows)
# calculate hit, precision, recall average across all users
all_users_hit = df_eval["Hit@5"].mean()
all_users_precision = df_eval["Precision@5"].mean()
all_users_recall = df_eval["Recall@5"].mean()

print("Metrics (averaged across all evaluated users):")
print("Hit@5 =", all_users_hit)
print("Precision@5 =", all_users_precision)
print("Recall@5 =", all_users_recall)

df_eval


Metrics (averaged across all evaluated users):
Hit@5 = 0.5
Precision@5 = 0.09999999999999999
Recall@5 = 0.5


/var/folders/sg/kf61262x7pzblj0zjwdr9ds00000gn/T/ipykernel_26971/584838127.py:30: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_reviews["datetime"] = pd.to_datetime(df_reviews["datetime"])


,user_id,test_movie,recommendations,Hit@5,Precision@5,Recall@5
0,1,Inception (2010),"[Inception (2010), Captain America: The Winter...",1,0.2,1.0
1,2,The Old Guard (2020),"[The Old Guard (2020), Inception (2010), Us (2...",1,0.2,1.0
2,3,The Old Guard (2020),"[Titanic (1997), Us (2019), The Dark Knight (2...",0,0.0,0.0
3,4,Captain America: The Winter Soldier (2014),"[Captain America: The Winter Soldier (2014), L...",1,0.2,1.0
4,5,Us (2019),"[Heat (1995), The Matrix (1999)]",0,0.0,0.0
5,6,The Old Guard (2020),"[Inception (2010), Captain America: The Winter...",0,0.0,0.0
6,7,Titanic (1997),"[Titanic (1997), Captain America: The Winter S...",1,0.2,1.0
7,8,Inception (2010),"[Titanic (1997), The Old Guard (2020), Coco (2...",0,0.0,0.0
8,9,Heat (1995),"[Titanic (1997), The Matrix (1999)]",0,0.0,0.0
9,10,Titanic (1997),"[Titanic (1997), Us (2019), The Dark Knight (2...",1,0.2,1.0


In [50]:
print("Metrics (averaged across all evaluated users):")
print("Hit@5 =", all_users_hit)
print("Precision@5 =", all_users_precision)
print("Recall@5 =", all_users_recall)

'''
I used the all-MiniLM-L6-v2 sentence transformer in order to create the embeddings
for each text review. Each movie review was stored as a vector representation.
Then, all of the review embeddings were aggregated by calculating the average
embedding vector for each unique movie. The movies that were rated were put in the
training set, with the most recent movie with a rating of 4 or above being the test set.
The training set was then used to create the user profile by averaging the embedding
vectors of these movies with a rating of 4 or above. Then, I found the cosine similarity
between the user profile and the candidate movies to generate recommendations (excluding
movies that were already seen). So, this used movies with ratings of 4 or above for the
user profile to make sure the recommendations were based off the user's preferred movies
so that the user would be more likely to enjoy them.
'''

print("Hit, Precision, Recall by user:")
df_eval

Metrics (averaged across all evaluated users):
Hit@5 = 0.5
Precision@5 = 0.09999999999999999
Recall@5 = 0.5
Hit, Precision, Recall by user:


,user_id,test_movie,recommendations,Hit@5,Precision@5,Recall@5
0,1,Inception (2010),"[Inception (2010), Captain America: The Winter...",1,0.2,1.0
1,2,The Old Guard (2020),"[The Old Guard (2020), Inception (2010), Us (2...",1,0.2,1.0
2,3,The Old Guard (2020),"[Titanic (1997), Us (2019), The Dark Knight (2...",0,0.0,0.0
3,4,Captain America: The Winter Soldier (2014),"[Captain America: The Winter Soldier (2014), L...",1,0.2,1.0
4,5,Us (2019),"[Heat (1995), The Matrix (1999)]",0,0.0,0.0
5,6,The Old Guard (2020),"[Inception (2010), Captain America: The Winter...",0,0.0,0.0
6,7,Titanic (1997),"[Titanic (1997), Captain America: The Winter S...",1,0.2,1.0
7,8,Inception (2010),"[Titanic (1997), The Old Guard (2020), Coco (2...",0,0.0,0.0
8,9,Heat (1995),"[Titanic (1997), The Matrix (1999)]",0,0.0,0.0
9,10,Titanic (1997),"[Titanic (1997), Us (2019), The Dark Knight (2...",1,0.2,1.0
